In [ ]:
!pip install monai SimpleITK lifelines scikit-survival -q

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from google.colab import drive

drive.mount('/content/drive')

BASE   = Path('/content/drive/MyDrive/multimodal_prognosis/')
TENSOR = BASE / 'tensors'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
#  dataset 
class CTSurvivalDataset(Dataset):
    def __init__(self, labels_csv, tensor_dir):
        self.df         = pd.read_csv(labels_csv)
        self.tensor_dir = Path(tensor_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        ct_tensor = torch.load(self.tensor_dir / f"{row['patient_id']}.pt").float()
        tabular   = torch.tensor([
            row['age']   / 100.0,
            row['stage'] / 4.0,
            float(row['gender']),
        ], dtype=torch.float32)
        time  = torch.tensor(row['survival_days'], dtype=torch.float32)
        event = torch.tensor(row['event'],         dtype=torch.float32)
        return ct_tensor, tabular, time, event


# encoders 
class ResBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels), nn.ReLU(inplace=True),
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))


class ImageEncoder(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(16), nn.ReLU(inplace=True), ResBlock3D(16),
            nn.Conv3d(16, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True), ResBlock3D(32),
            nn.Conv3d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True), ResBlock3D(64),
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
        )
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.proj(self.encoder(x))


class TabularEncoder(nn.Module):
    def __init__(self, in_dim=3, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(64, out_dim),
        )

    def forward(self, x):
        return self.net(x)


# loss and metric 
def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order]
    events      = events[order]
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    return -torch.mean((risk_scores - log_cumsum) * events)


def concordance_index(risk_scores, times, events):
    risk_scores = risk_scores.cpu().numpy()
    times       = times.cpu().numpy()
    events      = events.cpu().numpy()
    concordant = comparable = 0
    for i in range(len(times)):
        for j in range(len(times)):
            if events[i] == 1 and times[i] < times[j]:
                comparable += 1
                if risk_scores[i] > risk_scores[j]:
                    concordant += 1
                elif risk_scores[i] == risk_scores[j]:
                    concordant += 0.5
    return concordant / comparable if comparable > 0 else 0.5


# baseline models 
class ImageOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ImageEncoder(out_dim=64)
        self.head    = nn.Linear(64, 1)

    def forward(self, ct, _tab):
        return self.head(self.encoder(ct)).squeeze(-1)


class TabularOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TabularEncoder(in_dim=3, out_dim=64)
        self.head    = nn.Linear(64, 1)

    def forward(self, _ct, tab):
        return self.head(self.encoder(tab)).squeeze(-1)

In [ ]:
torch.manual_seed(42)

dataset  = CTSurvivalDataset(BASE / 'NSCLC-Radiomics-Lung1', TENSOR)
train_ds, val_ds, test_ds = random_split(dataset, [140, 30, 30])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
class ConcatFusionModel(nn.Module):
    """Simple baseline fusion — just concatenate image and tabular features."""
    def __init__(self):
        super().__init__()
        self.img_encoder = ImageEncoder(out_dim=64)
        self.tab_encoder = TabularEncoder(in_dim=3, out_dim=64)
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, ct, tab):
        f_img = self.img_encoder(ct)
        f_tab = self.tab_encoder(tab)
        fused = torch.cat([f_img, f_tab], dim=1)  # (B, 128)
        return self.head(fused).squeeze(-1)

In [ ]:
class CrossModalAttentionFusion(nn.Module):
    """
    Attention fusion: imaging features attend to tabular features.
    The model learns which clinical variables matter given what it sees in the scan.

    Mechanism:
        Q = W_q(f_img)          query from imaging
        K = W_k(f_tab)          key   from tabular
        V = W_v(f_tab)          value from tabular
        a = softmax(QKᵀ / √d)  attention weight
        f_attended = a * V      attended tabular context
        fused = [f_img ; f_attended]
    """
    def __init__(self, feat_dim=64, num_heads=4):
        super().__init__()
        self.img_encoder = ImageEncoder(out_dim=feat_dim)
        self.tab_encoder = TabularEncoder(in_dim=3, out_dim=feat_dim)

        # Projections for Q, K, V
        self.W_q = nn.Linear(feat_dim, feat_dim)
        self.W_k = nn.Linear(feat_dim, feat_dim)
        self.W_v = nn.Linear(feat_dim, feat_dim)

        self.scale = feat_dim ** 0.5

        # Also attend in reverse: tabular attends to imaging
        self.W_q2 = nn.Linear(feat_dim, feat_dim)
        self.W_k2 = nn.Linear(feat_dim, feat_dim)
        self.W_v2 = nn.Linear(feat_dim, feat_dim)

        # Final prediction head takes both attended features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, 64),
            nn.LayerNorm(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, ct, tab):
        f_img = self.img_encoder(ct)   # (B, D)
        f_tab = self.tab_encoder(tab)  # (B, D)

        # Direction 1: image attends to tabular
        Q1  = self.W_q(f_img).unsqueeze(1)   # (B, 1, D)
        K1  = self.W_k(f_tab).unsqueeze(1)
        V1  = self.W_v(f_tab).unsqueeze(1)
        a1  = torch.softmax(
                  torch.bmm(Q1, K1.transpose(1, 2)) / self.scale, dim=-1)
        f1  = torch.bmm(a1, V1).squeeze(1)   # (B, D)

        # Direction 2: tabular attends to image
        Q2  = self.W_q2(f_tab).unsqueeze(1)
        K2  = self.W_k2(f_img).unsqueeze(1)
        V2  = self.W_v2(f_img).unsqueeze(1)
        a2  = torch.softmax(
                  torch.bmm(Q2, K2.transpose(1, 2)) / self.scale, dim=-1)
        f2  = torch.bmm(a2, V2).squeeze(1)   # (B, D)

        # Combine both attended representations
        fused = torch.cat([f1, f2], dim=1)    # (B, 2D)
        return self.head(fused).squeeze(-1)

    def get_attention_weights(self, ct, tab):
        """Returns attention weights for interpretability analysis."""
        f_img = self.img_encoder(ct)
        f_tab = self.tab_encoder(tab)
        Q1    = self.W_q(f_img).unsqueeze(1)
        K1    = self.W_k(f_tab).unsqueeze(1)
        a1    = torch.softmax(
                    torch.bmm(Q1, K1.transpose(1, 2)) / self.scale, dim=-1)
        return a1.squeeze()   # (B,) — scalar per patient here


# Shape check
model_check = CrossModalAttentionFusion().to(DEVICE)
ct_dummy    = torch.randn(4, 1, 64, 64, 64).to(DEVICE)
tab_dummy   = torch.randn(4, 3).to(DEVICE)
out         = model_check(ct_dummy, tab_dummy)
print(f"Fusion output shape: {out.shape}")   # expect (4,)

In [ ]:
def train_model(model, train_loader, val_loader,
                epochs=50, lr=1e-3, save_path=None):

    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=5)

    history  = {'train_loss': [], 'val_cindex': []}
    best_ci  = 0.0

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for ct, tab, time, event in train_loader:
            ct, tab     = ct.to(DEVICE), tab.to(DEVICE)
            time, event = time.to(DEVICE), event.to(DEVICE)
            opt.zero_grad()
            risk = model(ct, tab)
            loss = cox_loss(risk, time, event)
            if torch.isnan(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
            n_batches  += 1

        model.eval()
        all_risk, all_time, all_event = [], [], []
        with torch.no_grad():
            for ct, tab, time, event in val_loader:
                ct, tab = ct.to(DEVICE), tab.to(DEVICE)
                all_risk.append(model(ct, tab).cpu())
                all_time.append(time)
                all_event.append(event)

        risk_cat  = torch.cat(all_risk)
        time_cat  = torch.cat(all_time)
        event_cat = torch.cat(all_event)
        val_ci    = concordance_index(risk_cat, time_cat, event_cat)

        sched.step(val_ci)
        history['train_loss'].append(epoch_loss / max(n_batches, 1))
        history['val_cindex'].append(val_ci)

        if val_ci > best_ci and save_path:
            best_ci = val_ci
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {epoch_loss/max(n_batches,1):.4f} "
                  f"| Val C-index: {val_ci:.4f} | Best: {best_ci:.4f}")

    if save_path and Path(save_path).exists():
        model.load_state_dict(torch.load(save_path))

    return history, best_ci

In [ ]:
print("=" * 50)
print("Training concat fusion")
print("=" * 50)

concat_model = ConcatFusionModel().to(DEVICE)
history_concat, best_concat_ci = train_model(
    concat_model, train_loader, val_loader,
    epochs=50, lr=1e-3,
    save_path=BASE / 'concat_fusion_best.pt'
)
print(f"\nConcat fusion best val C-index: {best_concat_ci:.4f}")

In [ ]:
print("=" * 50)
print("Training cross-modal attention fusion")
print("=" * 50)

attn_model = CrossModalAttentionFusion().to(DEVICE)
history_attn, best_attn_ci = train_model(
    attn_model, train_loader, val_loader,
    epochs=50, lr=1e-3,
    save_path=BASE / 'attn_fusion_best.pt'
)
print(f"\nAttention fusion best val C-index: {best_attn_ci:.4f}")

In [ ]:
def evaluate_on_test(model, test_loader):
    model.eval()
    all_risk, all_time, all_event = [], [], []
    with torch.no_grad():
        for ct, tab, time, event in test_loader:
            ct, tab = ct.to(DEVICE), tab.to(DEVICE)
            all_risk.append(model(ct, tab).cpu())
            all_time.append(time)
            all_event.append(event)
    return concordance_index(
        torch.cat(all_risk),
        torch.cat(all_time),
        torch.cat(all_event)
    )


# Load saved baselines 
image_model = ImageOnlyModel().to(DEVICE)
image_model.load_state_dict(torch.load(BASE / 'image_only_best.pt'))

tab_model = TabularOnlyModel().to(DEVICE)
tab_model.load_state_dict(torch.load(BASE / 'tabular_only_best.pt'))

# Evaluate all four
results = {
    'Image-only'      : evaluate_on_test(image_model,   test_loader),
    'Tabular-only'    : evaluate_on_test(tab_model,     test_loader),
    'Concat fusion'   : evaluate_on_test(concat_model,  test_loader),
    'Attention fusion': evaluate_on_test(attn_model,    test_loader),
}

print("=" * 50)
print("FINAL ABLATION TABLE — test C-index")
print("=" * 50)
for name, ci in results.items():
    bar   = '█' * int(ci * 40)
    delta = ci - results['Tabular-only']
    sign  = '+' if delta >= 0 else ''
    print(f"{name:<22} {ci:.4f}  {bar}  ({sign}{delta:.4f} vs tabular-only)")

print()
print(f"Best model: {max(results, key=results.get)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, 51)

colors = {
    'Concat fusion'   : 'mediumseagreen',
    'Attention fusion': 'mediumpurple',
}

for name, hist, color in [
    ('Concat fusion',    history_concat, 'mediumseagreen'),
    ('Attention fusion', history_attn,   'mediumpurple'),
]:
    axes[0].plot(epochs, hist['train_loss'], color=color, label=name)
    axes[1].plot(epochs, hist['val_cindex'], color=color, label=name)

# baseline references on C-index plot
axes[1].axhline(0.5,                      color='gray',      linestyle='--', alpha=0.5, label='Random (0.5)')
axes[1].axhline(results['Image-only'],    color='steelblue', linestyle=':',  alpha=0.7, label=f"Image-only test={results['Image-only']:.3f}")
axes[1].axhline(results['Tabular-only'],  color='coral',     linestyle=':',  alpha=0.7, label=f"Tabular-only test={results['Tabular-only']:.3f}")
axes[1].axhline(results['Attention fusion'], color='mediumpurple', linestyle='-', alpha=0.4, label=f"Attn test={results['Attention fusion']:.3f}")

axes[0].set_title('Training loss — fusion models')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cox loss')
axes[0].legend()

axes[1].set_title('Validation C-index — all models')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('C-index')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0.3, 1.0)

plt.tight_layout()
plt.savefig(BASE / 'fusion_results.png', dpi=150, bbox_inches='tight')
plt.show()